In [11]:
pip install segment-anything

In [12]:
# Importing necessary modules from the segment_anything package for SAM model
from segment_anything import sam_model_registry, SamPredictor

# Importing essential libraries for deep learning and visualization
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt

In [13]:
!wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth

--2026-09-08 18:18:34--  https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 54.240.184.91, 54.240.184.92, 54.240.184.75, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|54.240.184.91|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2564550879 (2.4G) [binary/octet-stream]
Saving to: ‘sam_vit_h_4b8939.pth.2’

sam_vit_h_4b8939.pt  10%[=>                  ] 246.82M  94.8MB/s               ^C


In [14]:
# Assuming you have the necessary imports
image_path = '/content/Screenshot 2026-09-08 223328.png'
image = plt.imread(image_path)  # Read the image using Matplotlib

# Convert image to RGB if it has an alpha channel and scale to 0-255
if image.shape[-1] == 4: # Check if it's RGBA
    image = image[..., :3] # Drop the alpha channel
if image.max() <= 1.0: # Check if values are in [0, 1] range
    image = (image * 255).astype(np.uint8) # Scale to [0, 255] and convert to uint8

plt.imshow(image)  # Display the image using Matplotlib
plt.xticks([])  # Remove x-axis ticks
plt.yticks([])  # Remove y-axis ticks
plt.show()  # Show the image plot

In [15]:
# Load pre-trained model checkpoint for Vision Transformer model from a web address
sam_checkpoint = "sam_vit_h_4b8939.pth" # In case you are not utilizing Google Colab, it is recommended that you obtain the SAM checkpoints from this web address:
                                        #https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
# Specify the type of Vision Transformer model to use
model_type = "vit_h"
# Specify the device to use for computation (CPU or GPU)
device = "cpu" # If you have gpu on your computer and is campatibale with torch you cand change "cpu" to "cuda".
                # In order to check torch is compatible with torch in your system you can use torch.cuda.is_available().
                # If it returns True, you can use cuda.

# Initialize segmentation model using a factory function with the loaded checkpoint as an argument
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
# Move the segmentation model to the specified device
sam.to(device=device)
# Create predictor object for the segmentation model
predictor = SamPredictor(sam)

In [16]:
# Set the image for prediction using the SAM predictor object
predictor.set_image(image)

In [17]:
# IPython magic command to configure how plots are displayed
%matplotlib
%matplotlib

Using matplotlib backend: agg
Using matplotlib backend: agg


In [18]:
# Initialize empty lists to store clicked points and their labels
# Pre-adding some points for demonstration on the football field.
# You can modify these points or add more if you wish.
points = [
    [500, 500], # Center of the field
    [300, 400], # Another point on the field
    [700, 600]  # Yet another point on the field
]
labels = [
    1, # Foreground
    1, # Foreground
    1  # Foreground
]

# The interactive onclick function will still be defined, but not immediately used
# if you proceed with these pre-defined points.
def onclick(event): # Define a function to handle mouse clicks on the image
    if event.key == 'A': # Check if the key pressed is 'A'
        # Access global variables for points and labels
        global points
        global labels

        x = int(round(event.xdata)) # Extract x coordinate of the click
        y = int(round(event.ydata)) # Extract y coordinate of the click

        ax.plot(x, y, 'o', markersize=4, color='red') # Plot a red circle at the clicked point on the image
        plt.show() # Display the updated plot
        # Store the coordinates in the points list along with a label (in this case, the label is set to 1)
        points.append([x, y])
        labels.append(1)

In [19]:
# Create a subplot
ax = plt.subplot(111)
# Display the image
plt.imshow(image)
plt.xticks([])
plt.yticks([])
# Connect the mouse click event to the 'onclick()' function
cid = plt.gcf().canvas.mpl_connect('button_press_event', onclick)
# Show the plot
plt.show()

In [20]:
# Convert the lists of clicked points and labels into NumPy arrays
points = np.array(points)
labels = np.array(labels)

# Re-set the image for prediction using the SAM predictor object, as it might have been reset by plot interaction.
predictor.set_image(image)

# Use the predictor object to predict masks, scores, and logits
# The 'multimask_output=True' argument indicates that the model should generate multiple masks
masks, scores, logits = predictor.predict(
    point_coords=points,
    point_labels=labels,
    multimask_output=True,
)

In [21]:
def show_mask(mask, ax, random_color=True):
    # Check if random color is requested
    if random_color:
        # Generate a random color with alpha value for the mask
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        # Use a default shade of blue with alpha value for the mask
        color = np.array([30/255, 144/255, 255/255, 0.6])
    # Get the height and width of the mask
    h, w = mask.shape[-2:]
    # Reshape the mask into an RGB image by multiplying it element-wise with the color array
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    # Display the colored mask on the provided matplotlib axis
    ax.imshow(mask_image)

In [23]:
import os

# Create a directory to save segmented images if it doesn't exist
output_dir = "segmented_images"
os.makedirs(output_dir, exist_ok=True)

# Iterate over pairs of masks and scores using enumerate and zip
for i, (mask, score) in enumerate(zip(masks, scores)):

    fig, ax = plt.subplots(figsize=(10, 10)) # Create a new matplotlib figure and axes

    ax.imshow(image) # Display the original image on the axes

    # Use the show_mask function to display the current mask on the same figure
    show_mask(mask, ax)

    ax.set_title(f"Mask {i+1}, Score: {score:.3f}", fontsize=18) # Set the title including the mask index and its score

    ax.axis('off') # Turn off the axis for a cleaner appearance

    # Save the figure
    filename = os.path.join(output_dir, f"segmented_image_{i+1}.png")
    plt.savefig(filename, bbox_inches='tight', pad_inches=0)
    print(f"Saved: {filename}")

    # Show the figure (optional, you can remove plt.show() if you only want to save)
    plt.show()

Saved: segmented_images/segmented_image_1.png
Saved: segmented_images/segmented_image_2.png
Saved: segmented_images/segmented_image_3.png
